1. Import Libraries

In [46]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

2. Load Dataset

In [47]:
df = pd.read_csv("/content/tweets.csv")

# Display first rows
df.head()

,id,label,tweet
0,1,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,2,0,Finally a transparant silicon case ^^ Thanks t...
2,3,0,We love this! Would you go? #talk #makememorie...
3,4,0,I'm wired I know I'm George I was made that wa...
4,5,1,What amazing service! Apple won't even talk to...


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7920 entries, 0 to 7919
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      7920 non-null   int64 
 1   label   7920 non-null   int64 
 2   tweet   7920 non-null   object
dtypes: int64(2), object(1)
memory usage: 185.8+ KB


3. Text Cleaning

In [49]:
def clean_text(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions, hashtags
    text = re.sub(r"@\w+|#", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = text.strip()

    return text

df['clean_text'] = df['tweet'].apply(clean_text)

df[['tweet', 'clean_text']].head()

,tweet,clean_text
0,#fingerprint #Pregnancy Test https://goo.gl/h1...,fingerprint pregnancy test android apps beaut...
1,Finally a transparant silicon case ^^ Thanks t...,finally a transparant silicon case thanks to ...
2,We love this! Would you go? #talk #makememorie...,we love this would you go talk makememories un...
3,I'm wired I know I'm George I was made that wa...,im wired i know im george i was made that way ...
4,What amazing service! Apple won't even talk to...,what amazing service apple wont even talk to m...


4. Features & Labels

In [50]:
X = df['clean_text']
y = df['label']

5. Train-Test Split

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

6. TF-IDF Vectorization

In [52]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

7. Train Model

In [53]:
model = LogisticRegression(class_weight='balanced', max_iter=200)

model.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=200)

8. Predictions

In [54]:
y_pred = model.predict(X_test_vec)

9. Evaluation

In [55]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8958333333333334

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.90      0.93      1152
           1       0.77      0.88      0.82       432

    accuracy                           0.90      1584
   macro avg       0.86      0.89      0.87      1584
weighted avg       0.90      0.90      0.90      1584



10. Test Your Own Tweet

In [56]:
def predict_sentiment(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)

    return "Negative" if prediction[0] == 1 else "Positive"


# Try examples
print(predict_sentiment("I love this phone, amazing performance!"))
print(predict_sentiment("Worst mobile ever, waste of money"))

Positive
Negative


Initially, the model achieved an accuracy of 87.4%, performing particularly well at identifying negative sentiments, with a high recall of 96%. However, it struggled to detect positive tweets effectively, with a lower recall of 64%, likely due to the imbalance in the dataset where negative samples were more frequent. After applying class balancing, the model showed noticeable improvement. The accuracy increased to 89.6%, and more importantly, the recall for positive sentiments rose significantly to 88%.